In [0]:
import argparse
import logging
import os

import sys
sys.path.append("..")
sys.path.append("../..")


import lib_etl.validations_ETL as validations
from lib.s3 import etl_input_table_validator
from lib.job_manager import load_config, split_config


In [0]:
%run ../../config/utils

In [0]:
run_as_date = dbutils.widgets.get("run_as_date")

In [0]:


def filter_gas_nr(detail_fiscal):
    """
    Filters the detail table for only positive sales (no returns).
    And filters out transactions at BJs cafe and other minor exclusions
    (established by the client).
    """
    mc_cds = ["402030190", "402030191", "203010098"]

    return detail_fiscal.filter(
        (detail_fiscal.EXTENDED_PRC_AMT > 0)
        & (detail_fiscal.QTY_IN_UNITS > 0)
        & (~detail_fiscal.MC_CD.isin(mc_cds))
    )



In [0]:


def main(data_paths, config_validation):

    logging.info("Starting processing table detail_gas_nr_fiscal")

    print("Calling data_paths.......")
    recency_lookback_duration = data_paths.get("recency_lookback_duration", {})

    print("************************")
    print(recency_lookback_duration)
    print("************************")

    print("Calling  etl_input_data_validator.......")
    
    etl_input_table_validator(
        silver_transaction_fiscal_detail,
        recency_lookback_duration=recency_lookback_duration,
        spark=spark
    )

    detail_fiscal = spark.table(silver_transaction_fiscal_detail)

    detail_gas_nr_fiscal = filter_gas_nr(detail_fiscal)
    detail_gas_nr_fiscal.createOrReplaceTempView("source")

    validations.validate_table(
        spark,
        "intermediate",
        "detail_gas_nr_fiscal",
        config_validation,
        detail_gas_nr_fiscal,
        stats_etl_path
    )

    logging.info(
        "Saving the intermediate file "
        + silver_transaction_fiscal_detail_gas_nr
    )
    
    detail_gas_nr_fiscal.write.mode("overwrite").saveAsTable(
        silver_transaction_fiscal_detail_gas_nr
    )

    if archive_flag:
        save_archive(detail_gas_nr_fiscal, silver_transaction_fiscal_detail_gas_nr_archive, run_as_date)
    # (detail_gas_nr_fiscal.repartition("FISCAL_WEEK_END")
    #  .write
    #  .format("delta")
    #  .mode("overwrite")
    #  .partitionBy("FISCAL_WEEK_END")
    #  .saveAsTable(silver_transaction_fiscal_detail_gas_nr))

    print("detail_fiscal schema")
    detail_fiscal.printSchema()
    print("detail_gas_nr_fiscal schema")
    detail_gas_nr_fiscal.printSchema()


In [0]:

parser = argparse.ArgumentParser()

try:
    base_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    base_dir = os.getcwd()

default_config_path = os.path.join(base_dir, "../config/config.yaml")

print(f"base_dir: {base_dir}")
print(f"default_config_path: {default_config_path}")

parser.add_argument(
   "--config_path",
    type=str,
    default=default_config_path,
    help=(
        """
        path to the config file
        """
    ),
)

try:
    print('Parsing arguments...')
    args, unknown = parser.parse_known_args()

    print('Loading config...')
    config = load_config(args.config_path)

    print('Splitting config...')
    data_paths, club_square_config, config_validation = split_config(config)

    print('Running main...')
    main(data_paths, config_validation)

except Exception as e:
    print("Failure occurred:", e)

